# Train All Models

This notebook pulls the triggers for all tasks on all notebooks responsible for model creation and training.

Note that this notebook also has a parametrized cell and can be executed with Papermill using the following code snipet:

```python
import papermill as pm
from nbformat import NotebookNode

execution_result: NotebookNode = pm.execute_notebook(
    "2 - train_all_models.ipynb",                       # adjust the path if you are not running from the same directory
    None,                                           # path of the copy of the executed notebook, if you need it
    parameters = dict(
        config_file_path=<config_file_path>,        # the path of the config file
        data_file_path=<data_file_path>,            # the path of the generated CSV file
        mlflow_tracking_uri=<mlflow_tracking_uri>,  # a valid tracking uri
    )
)
```

## Preparation

In [1]:
import json
import mlflow
import papermill as pm
from nbformat import NotebookNode
from utils import print_system_info

print_system_info()

Python version: 3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Operating System: Darwin 23.4.0
Platform: macOS-14.4-arm64-arm-64bit
Processor: arm
Machine: arm64
CPU count: 11


In [2]:
config_file_path = "config_rhs.json"
data_file_path = "data_rhs.csv"
mlflow_tracking_uri = "sqlite:///mlflow.db"

In [3]:
with open(config_file_path, "r") as f:
    config: dict = json.load(f)
    
section_type = config["section"]["type"]
print(f"Section type from config: {section_type}")

mlflow.set_tracking_uri(mlflow_tracking_uri)

Section type from config: rectangular_hollow_section


In [4]:
# from utils.ml import delete_all_logged_models
# delete_all_logged_models()

## Train utilitization estimator models

The goal here is to learn the utilization function for a particular section and a set of internal forces.

In [5]:
task="utilization_estimation"
experiment_name = f"{task}__{section_type}"
experiment = mlflow.set_experiment(experiment_name)
mlflow.set_experiment_tag("section_type", section_type)
mlflow.set_experiment_tag("task", task)

2025/11/13 16:33:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/13 16:33:42 INFO mlflow.store.db.utils: Updating database tables
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025-11-13 16:33:42 INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025-11-13 16:33:42 INFO  [alembic.runtime.mig

In [6]:
execution_result: NotebookNode = pm.execute_notebook(
    "2a - train_sklearn_utilization_estimators.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.725e+02, tolerance: 6.487e-01
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.747e+02, tolerance: 6.294e-01
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/s

In [7]:
execution_result: NotebookNode = pm.execute_notebook(
    "2f - train_ANN_utilization_estimator.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

## Train failure predictor models

These are classification models with the goal of predictiong the failure of a particular cross section for a given set of internal forces. Hence, the task here is binary classification.

In [8]:
task="failure_prediction"
experiment_name = f"{task}__{section_type}"
experiment = mlflow.set_experiment(experiment_name)
mlflow.set_experiment_tag("section_type", section_type)
mlflow.set_experiment_tag("task", task)

2025/11/13 16:46:01 INFO mlflow.tracking.fluent: Experiment with name 'failure_prediction__rectangular_hollow_section' does not exist. Creating a new experiment.


In [9]:
execution_result: NotebookNode = pm.execute_notebook(
    "2b - train_sklearn_failure_predictors.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/22 [00:00<?, ?cell/s]

In [10]:
execution_result: NotebookNode = pm.execute_notebook(
    "2d - train_ANN_failure_predictor.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/36 [00:00<?, ?cell/s]

## Train geometry validator models

The goal is to tell if the parameters of a section define admit a valid geometry or not.

In [11]:
task="geometry_validation"
experiment_name = f"{task}__{section_type}"
experiment = mlflow.set_experiment(experiment_name)
mlflow.set_experiment_tag("section_type", section_type)
mlflow.set_experiment_tag("task", task)

2025/11/13 16:47:55 INFO mlflow.tracking.fluent: Experiment with name 'geometry_validation__rectangular_hollow_section' does not exist. Creating a new experiment.


In [12]:
execution_result: NotebookNode = pm.execute_notebook(
    "2g - train_sklearn_geometry_validators.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

## Train section property estimator models

The goal for these models is to approximate the properties of a cross section like cross sectional area, stiffness, etc.

In [13]:
task="section_estimation"
experiment_name = f"{task}__{section_type}"
experiment = mlflow.set_experiment(experiment_name)
mlflow.set_experiment_tag("section_type", section_type)
mlflow.set_experiment_tag("task", task)

2025/11/13 16:48:23 INFO mlflow.tracking.fluent: Experiment with name 'section_estimation__rectangular_hollow_section' does not exist. Creating a new experiment.


In [14]:
execution_result: NotebookNode = pm.execute_notebook(
    "2c - train_sklearn_section_estimators.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/19 [00:00<?, ?cell/s]

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.795e+26, tolerance: 6.072e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.397e+26, tolerance: 6.140e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/s

In [15]:
execution_result: NotebookNode = pm.execute_notebook(
    "2e - train_ANN_section_estimator.ipynb",
    None,
    parameters = dict(
        config_file_path=config_file_path,
        data_file_path=data_file_path,
        mlflow_experiment_name=experiment_name,
        mlflow_tracking_uri=mlflow_tracking_uri,
        task=task,
    )
)

Executing:   0%|          | 0/26 [00:00<?, ?cell/s]